# 05 Synthetic Sensor Research (Phase 5)

**Synthetic RAW 50MP Lab.** A secondary research branch (PRD sections 14 and 30, Phase 5) that turns a
processed linear RGB master into **synthetic** sensor-domain data and measures what that does and does not add:

- assumed camera colour transformation, synthetic exposure scaling and synthetic CFA (Bayer) sampling,
- optional shot and read noise from a stated hypothetical sensor model, quantisation to the ADC bit depth,
- a research DNG that is written only if it passes structural validation and opens in LibRaw,
- experiment **E08**: demosaic the synthetic mosaic back to RGB and measure the round trip with the Phase 1
  metrics,
- the PRD 14.5 bit-depth tests and experiment **E01**: does expanding the bit depth add information?

All processing code lives in the shared `synthetic_raw_lab` package; this notebook only sets parameters, calls
it and displays results.

## 1. Introduction and scientific limitations

- Converting RGB into a Bayer-like mosaic **simulates** sensor sampling. It does not reconstruct the samples a
  real camera would have measured, and simulated shot noise is not evidence of a physical exposure (PRD 14.3).
- Every mosaic, DNG and developed image is labelled **synthetic** in its data, file metadata, sidecar, manifest
  and report. None of it is camera-native RAW (PRD 17.3).
- The input master is linear-light display-referred data from an AI render, not calibrated scene radiance.
- More bits per sample add numerical precision, not original scene information; section 15 measures this.

## 2. Runtime setup

Set the run parameters. In Google Colab these are form fields.

- **Private repository:** add a Colab secret named `GITHUB_TOKEN` (read-only) so the setup cell can clone the
  repository, or copy it to Google Drive and set `PROJECT_DIR`.
- **Input:** `master` reads an existing linear OpenEXR master (`MASTER_PATH`, for example a Phase 0/1 Lanczos
  master on Drive). `upload`, `path` and `synthetic` take a source image instead; the notebook then builds the
  master with the Phase 0/1 Lanczos baseline first (`MASTER_PRESET`: `quick` makes a reduced master, `standard`
  the full 50 MP) and also runs E01 on the source codes.
- **Keep outputs:** a 50 MP run writes about 1.5 GB (mosaic TIFF, DNG, bit-depth files). Set `MOUNT_DRIVE` and
  point `OUTPUT_ROOT` at Drive to keep them.
- `CONFIG_FILE` holds every sensor parameter (PRD 14.4); the fields below override the most used ones.

In [ ]:
# @title Run parameters
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
PROJECT_DIR = ""  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
OUTPUT_ROOT = ""  # @param {type:"string"}
REPORTS_ROOT = ""  # @param {type:"string"}
INPUTS_ROOT = ""  # @param {type:"string"}
CONFIG_FILE = "configs/sensor_research.yaml"  # @param {type:"string"}
BASELINE_CONFIG_FILE = "configs/baseline_default.yaml"  # @param {type:"string"}
INPUT_MODE = "master"  # @param ["master", "upload", "path", "synthetic"]
MASTER_PATH = ""  # @param {type:"string"}
INPUT_PATH = ""  # @param {type:"string"}
SYNTHETIC_SIZE = "3840x2160"  # @param {type:"string"}
CATEGORY = "universal"  # @param ["human", "night", "universal"]
MASTER_PRESET = "standard"  # @param ["quick", "standard"]
QUICK_MEGAPIXELS = 2.0  # @param {type:"number"}
CFA_PATTERN = "RGGB"  # @param ["RGGB", "BGGR", "GRBG", "GBRG"]
ADC_BIT_DEPTH = 14  # @param [10, 12, 14, 16]
ANALOG_GAIN = 1.0  # @param {type:"number"}
SHOT_NOISE = False  # @param {type:"boolean"}
READ_NOISE = False  # @param {type:"boolean"}
EXPORT_SYNTHETIC_DNG = True  # @param {type:"boolean"}
INSTALL_EXIFTOOL = True  # @param {type:"boolean"}

## 3. Dependency installation

Finds the repository (`PROJECT_DIR`, the checkout this notebook lives in, or the Colab clone of `REPO_REF`) and
installs the `synthetic_raw_lab` package with its dependencies, including `rawpy` (LibRaw) for the DNG reader
check. In Colab, `INSTALL_EXIFTOOL` also installs ExifTool so the DNG gets an independent `-validate` check.

In [ ]:
import base64
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

NOTEBOOK_VERSION = "05_Synthetic_Sensor_Research/1.0"
COLAB_CLONE = Path("/content/experiment_50MP")
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project() -> Path:
    if PROJECT_DIR:
        return Path(PROJECT_DIR).expanduser().resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pipeline.py").exists():
            return candidate.resolve()
    return COLAB_CLONE


def git(*args: str) -> None:
    env = dict(os.environ)
    token = ""
    if IN_COLAB:
        try:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN") or ""
        except Exception:
            token = ""
    if token:
        credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {credential}",
        )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"git failed (is GITHUB_TOKEN set for this private repository?): {result.stderr}")


PROJECT = find_project()
if not (PROJECT / "pyproject.toml").exists():
    git("clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT))
elif PROJECT == COLAB_CLONE:
    git("-C", str(PROJECT), "fetch", "--depth", "1", REPO_URL, REPO_REF)
    git("-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD")
spec = importlib.util.find_spec("synthetic_raw_lab")
if spec is None or Path(spec.origin).resolve().parent != PROJECT / "src":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(PROJECT)], check=True)
    importlib.invalidate_caches()
for name in [m for m in sys.modules if m.partition(".")[0] == "synthetic_raw_lab"]:
    del sys.modules[name]

import synthetic_raw_lab  # noqa: E402
from synthetic_raw_lab import environment  # noqa: E402

print(f"project: {PROJECT} (Git: {environment.git_revision(PROJECT)})")
print(f"synthetic_raw_lab {synthetic_raw_lab.__version__} from {Path(synthetic_raw_lab.__file__).parent}")

if INSTALL_EXIFTOOL and IN_COLAB and shutil.which("exiftool") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "libimage-exiftool-perl"], check=False)
print(f"exiftool: {shutil.which('exiftool') or 'not installed (that DNG check is skipped; LibRaw still runs)'}")

## 4. GPU and environment inspection

Loads the sensor configuration (`CONFIG_FILE` plus the form overrides) and, for source inputs, the Phase 0/1
configuration that builds the master. Phase 5 runs on the CPU; no GPU is needed.

In [ ]:
from IPython.display import Image, Markdown, display

from synthetic_raw_lab import config as lab_config
from synthetic_raw_lab import (
    environment,
    ingestion,
    pipeline,
    sensor_config,
    sensor_research,
    test_images,
)


def project_path(path: str) -> Path:
    return Path(path) if Path(path).is_absolute() else PROJECT / path


paths = {"output_root": OUTPUT_ROOT, "reports_root": REPORTS_ROOT}
SENSOR_CONFIG = sensor_config.load_sensor_config(
    project_path(CONFIG_FILE),
    input_category=CATEGORY,
    cfa_pattern=CFA_PATTERN,
    adc_bit_depth=int(ADC_BIT_DEPTH),
    analog_gain=float(ANALOG_GAIN),
    shot_noise_enabled=SHOT_NOISE,
    read_noise_enabled=READ_NOISE,
    export_synthetic_dng=EXPORT_SYNTHETIC_DNG,
    **{name: value for name, value in paths.items() if value},
)
BASELINE_CONFIG = lab_config.load_config(
    project_path(BASELINE_CONFIG_FILE),
    input_category=CATEGORY,
    runtime_preset=MASTER_PRESET,
    quick_target_megapixels=float(QUICK_MEGAPIXELS),
    baselines=["lanczos"],
    **{name: value for name, value in {**paths, "inputs_root": INPUTS_ROOT}.items() if value},
)
INPUTS_DIR, OUTPUT_DIR, REPORTS_DIR = pipeline.resolve_roots(BASELINE_CONFIG, PROJECT)

runtime = environment.inspect_runtime(OUTPUT_DIR)
print(f"Python {runtime['python']} on {runtime['platform']} (Colab: {runtime['in_colab']})")
print(f"GPU: {runtime['gpu']['name'] if runtime['gpu'] else 'none detected (not needed for Phase 5)'}")
print(f"rawpy (LibRaw): {runtime['packages'].get('rawpy', 'not installed')}")
print(
    f"RAM: {runtime['ram_available_gb']} of {runtime['ram_total_gb']} GB available; "
    f"free disk: {runtime['disk_free_gb']} GB; CPUs: {runtime['cpu_count']}"
)

## 5. Input upload

`master` uses `MASTER_PATH`. `upload` asks for a source image in Colab, `path` uses `INPUT_PATH`, and `synthetic`
generates the deterministic test image (`SYNTHETIC_SIZE`).

In [ ]:
STAGING = OUTPUT_DIR / "_staging"
SOURCE = None
if INPUT_MODE == "master":
    if not MASTER_PATH:
        raise ValueError("set MASTER_PATH to a linear OpenEXR master, or choose a source input mode")
    MASTER = Path(MASTER_PATH).expanduser()
elif INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("upload only works in Colab; use INPUT_MODE='path' or 'synthetic'")
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"upload exactly one image, got {len(uploaded)}")
    name, data = next(iter(uploaded.items()))
    STAGING.mkdir(parents=True, exist_ok=True)
    SOURCE = STAGING / name
    SOURCE.write_bytes(data)
elif INPUT_MODE == "path":
    SOURCE = Path(INPUT_PATH).expanduser()
elif INPUT_MODE == "synthetic":
    width, height = (int(v) for v in SYNTHETIC_SIZE.lower().split("x"))
    SOURCE = test_images.write_test_image(STAGING / f"synthetic_{width}x{height}.png", width, height)
else:
    raise ValueError(f"unknown INPUT_MODE {INPUT_MODE!r}")
print(f"master: {MASTER}" if SOURCE is None else f"source: {SOURCE} ({SOURCE.stat().st_size:,} bytes)")

## 6. Input validation

A source image is ingested and validated exactly as in Phase 0/1 (PRD 3.4); its decoded codes feed E01. A master
is read as linear RGB (converted to linear Rec.709 if its chromaticities differ) and its lineage attributes are
shown.

In [ ]:
SOURCE_CODES = None
if SOURCE is not None:
    source_image = ingestion.ingest(SOURCE, INPUTS_DIR, CATEGORY, source_color_space=BASELINE_CONFIG.source_color_space)
    v = source_image.validation
    SOURCE_CODES = source_image.codes
    print(
        f"{v['file']['format']} {v['dimensions']['width']} x {v['dimensions']['height']}, "
        f"{v['samples']['bit_depth']}-bit (effective {v['samples']['effective_bit_depth']}-bit), "
        f"{v['color']['resolved']['description']}"
    )
    for warning in source_image.warnings:
        print("WARNING:", warning)
    del source_image
else:
    master_input = sensor_research.read_linear_master(MASTER)
    print(f"{master_input.name}: {master_input.linear.shape[1]} x {master_input.linear.shape[0]} linear RGB")
    for key, value in master_input.lineage.items():
        print(f"  {key}: {value}")
    for note in master_input.notes:
        print("NOTE:", note)
    del master_input

## 7. Experiment configuration

Every Phase 5 parameter with its accepted values, effect and warnings (PRD 14.4, 22.2). The sensor values describe
a hypothetical sensor: they are stated assumptions, not a real camera.

In [ ]:
table = ["| parameter | value | default | accepted | effect | warning |", "|---|---|---|---|---|---|"]
for row in sensor_config.parameter_table(SENSOR_CONFIG):
    table.append(
        f"| `{row['parameter']}` | `{row['current']}` | `{row['default']}` | {row['accepted']} | "
        f"{row['effect']} | {row['warning']} |"
    )
display(Markdown("\n".join(table)))
model = SENSOR_CONFIG.sensor_model()
print(f"black/white level: {model.black}/{model.white}; conversion gain {model.dn_per_electron:.4f} DN/e-")
print(f"noise: {model.noise_description()}")

## 8. Scene analysis

Not part of Phase 5 (PRD Phase 4).

## 9. Baseline reconstruction

For a source input, builds the processed master with the Phase 0/1 pipeline (Lanczos baseline only, linear EXR
master). A `master` input skips this section.

In [ ]:
if SOURCE is not None:
    BASELINE_RESULT = pipeline.run(SOURCE, BASELINE_CONFIG, PROJECT, notebook_version=NOTEBOOK_VERSION)
    MASTER = Path(BASELINE_RESULT.manifest["baselines"]["lanczos"]["exports"]["exr"]["path"])
    print(f"Phase 0/1 run {BASELINE_RESULT.run_id}: master {MASTER.name}")
else:
    print(f"using the existing master {MASTER}")

## 10 to 12. Pretrained SR, specialised reconstruction, tonal and optical processing

Not part of Phase 5 (PRD Phases 2 and 3). Phase 5 can start from any later processed master by setting
`MASTER_PATH`.

## 13. Optional sensor simulation

Runs the Phase 5 experiments on the master: synthetic sensor simulation, mosaic TIFF and validated research DNG,
the E08 round trip with every demosaic method (and LibRaw from the DNG), and the bit-depth experiments. Every
stage is logged with its PRD 5.2 classification, duration and peak memory.

In [ ]:
try:
    RESULT = sensor_research.run(
        MASTER, SENSOR_CONFIG, PROJECT, source_codes=SOURCE_CODES, notebook_version=NOTEBOOK_VERSION
    )
except pipeline.StageError as error:
    print(f"FAILED in stage {error.stage}: {error}")
    if error.manifest_path:
        print(f"partial manifest and report: {error.manifest_path.parent}")
    print(
        "Reduced-memory retry: set INPUT_MODE to a source with MASTER_PRESET='quick', or turn off "
        "bit_depth_write_files in CONFIG_FILE, then re-run section 2 and every section after it."
    )
    raise
MANIFEST = RESULT.manifest
print(
    f"run {RESULT.run_id}: {MANIFEST['status']} in {MANIFEST['total_seconds']:.1f} s, "
    f"peak RSS {MANIFEST['memory']['pipeline_peak_rss_mb']:.0f} MB"
)

## 14. Professional exports

The simulated CFA data as a 16-bit greyscale TIFF and, when enabled, the research DNG. Each file was re-read and
checked; the DNG also had to open in LibRaw with the written CFA pattern, levels, white balance, colour matrix and a
bit-identical mosaic (PRD 14.6). Every file carries the synthetic label and has a JSON sidecar.

In [ ]:
rows = ["| file | kind | size | checks |", "|---|---|---|---|"]
for export in MANIFEST["exports"].values():
    passed = sum(export["checks"].values())
    rows.append(
        f"| `{Path(export['path']).name}` | {export['kind']} | {export['size_bytes'] / 1e6:.1f} MB | "
        f"{passed}/{len(export['checks'])} |"
    )
display(Markdown("\n".join(rows)))
if "dng" in MANIFEST["exports"]:
    exiftool = MANIFEST["exports"]["dng"]["exiftool"]
    print("ExifTool -validate:", "; ".join(exiftool["output"]))
print(f"label checks: {MANIFEST['labels']}")
print(f"outputs: {RESULT.outputs_dir}")

## 15. Quality metrics

**E08:** each demosaic method develops the synthetic mosaic back to linear Rec.709, compared with the master at
full resolution. **PRD 14.5 / E01:** quantisation error, headroom, gradient smoothness after an identical shadow
push, drift through repeated transformations and file size per representation, then whether expanding the source
bit depth adds information.

In [ ]:
from synthetic_raw_lab import sensor_report

roundtrip = MANIFEST["roundtrip"]
table = ["| metric | " + " | ".join(roundtrip) + " |", "|---" * (len(roundtrip) + 1) + "|"]
table += ["| " + " | ".join(str(c) for c in row) + " |" for row in sensor_report._roundtrip_rows(roundtrip)]
display(Markdown("\n".join(table)))

bit_depth = MANIFEST.get("bit_depth")
if bit_depth:
    names = [r["name"] for r in bit_depth["representations"]]
    table = ["| quantity | " + " | ".join(names) + " |", "|---" * (len(names) + 1) + "|"]
    table += ["| " + " | ".join(str(c) for c in row) + " |" for row in sensor_report._bit_depth_rows(bit_depth)]
    display(Markdown("\n".join(table)))
    if "expansion" in bit_depth:
        print("E01:", bit_depth["expansion"]["finding"])

## 16. Visual comparison

Identical crops at 100%: the synthetic CFA (each photosite in the colour it samples), the master, each round trip,
and the amplified difference between the master and the first method. Review at 100% and 200% for zipper
artefacts, false colour and noise (PRD 26.1).

In [ ]:
for panel in MANIFEST.get("crop_panels", []):
    print(f"crop {panel['crop']}")
    display(Image(filename=panel["path"]))

## 17. Report generation

The Markdown report and machine-readable manifest are written next to their figures (PRD 27).

In [ ]:
print(f"report:   {RESULT.report_path}")
print(f"manifest: {RESULT.manifest_path}")
display(Markdown("\n".join(f"- {line}" for line in sensor_report.conclusions(MANIFEST))))

## 18. Download and archive

Reports are archived separately from the mosaics and DNG (PRD 21.5). In Colab the report archive is downloaded;
the full-resolution files stay in `OUTPUT_ROOT`.

In [ ]:
archive = shutil.make_archive(str(RESULT.reports_dir), "zip", root_dir=RESULT.reports_dir)
print(f"report archive: {archive}")
if IN_COLAB:
    from google.colab import files

    files.download(archive)